<a href="https://colab.research.google.com/github/BirasaDivine/sdg3-indicator-text-classification-/blob/main/Exp8_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
import joblib
from scipy.sparse import load_npz, hstack
from sklearn.metrics import hamming_loss, f1_score, precision_score, recall_score
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
import warnings
warnings.filterwarnings('ignore')

SEED = 42
SAVE_DIR = '/content/drive/MyDrive/SDG_Assignment/processed_data'

# Labels
y_train = np.load(f'{SAVE_DIR}/y_train.npy')
y_val   = np.load(f'{SAVE_DIR}/y_val.npy')

# Features
X_train_tfidf    = load_npz(f'{SAVE_DIR}/X_train_tfidf.npz')
X_val_tfidf      = load_npz(f'{SAVE_DIR}/X_val_tfidf.npz')
X_train_combined = load_npz(f'{SAVE_DIR}/X_train_combined_tfidf.npz')
X_val_combined   = load_npz(f'{SAVE_DIR}/X_val_combined_tfidf.npz')
X_train_sbert    = np.load(f'{SAVE_DIR}/X_train_sbert.npy')
X_val_sbert      = np.load(f'{SAVE_DIR}/X_val_sbert.npy')

# Test features
X_test_tfidf    = load_npz(f'{SAVE_DIR}/X_test_tfidf.npz')
X_test_combined = load_npz(f'{SAVE_DIR}/X_test_combined.npz')
X_test_sbert_text = np.load(f'{SAVE_DIR}/X_test_text.npy', allow_pickle=True).tolist()
test_ids        = np.load(f'{SAVE_DIR}/test_ids.npy')

# Indicator names
ALL_INDICATORS = np.load(f'{SAVE_DIR}/indicator_names.npy', allow_pickle=True).tolist()
SHORT_NAMES    = [ind.split(' - ')[0] for ind in ALL_INDICATORS]

print('All data loaded')
print(f'Train: {y_train.shape[0]}  Val: {y_val.shape[0]}  Labels: {y_train.shape[1]}')

Mounted at /content/drive
All data loaded
Train: 2545  Val: 450  Labels: 27


In [5]:
# Load Exp 1-4 results from CSV
exp1_4 = pd.read_csv('/content/experiments_1_4_results (1).csv')

# Exp 5-7 results — hardcode from your teammate's notebook
exp5_7 = pd.read_csv('/content/experiments_5_7_results.csv')

all_results = pd.concat([exp1_4[['Exp','Name','Hamming Loss']], exp5_7], ignore_index=True)
print('=== All experiments so far ===')
print(all_results.sort_values('Hamming Loss').to_string(index=False))

=== All experiments so far ===
 Exp                                       Name  Hamming Loss Experiment       Feature Set                       Model  Threshold Strategy  Micro F1  Macro F1  Micro Precision  Micro Recall  Macro Precision  Macro Recall
 NaN                                        NaN      0.044362      Exp 5  Word+Char TF-IDF            OneVsRest LogReg Per-label 0.30-0.50  0.620155  0.404186         0.813309      0.501139         0.701546      0.314895
 3.0 Combined TF-IDF (Word + Char) + Linear SVM      0.044900        NaN               NaN                         NaN                 NaN       NaN       NaN              NaN           NaN              NaN           NaN
 2.0                   TF-IDF Word + Linear SVM      0.046400        NaN               NaN                         NaN                 NaN       NaN       NaN              NaN           NaN              NaN           NaN
 4.0         Linear SVM + class_weight=balanced      0.048300        NaN             

In [6]:
# Standardise both into the same 3 columns
exp1_4_clean = exp1_4[['Exp', 'Name', 'Hamming Loss']].copy()
exp1_4_clean.columns = ['Exp', 'Name', 'Hamming Loss']

exp5_7_clean = exp5_7[['Experiment', 'Model', 'Hamming Loss']].copy()
exp5_7_clean.columns = ['Exp', 'Name', 'Hamming Loss']

# Combine
all_results = pd.concat([exp1_4_clean, exp5_7_clean], ignore_index=True)
all_results = all_results.dropna(subset=['Hamming Loss'])
all_results = all_results.sort_values('Hamming Loss').reset_index(drop=True)

print('=== All experiments so far (sorted by Hamming Loss) ===')
print(all_results.to_string(index=False))

=== All experiments so far (sorted by Hamming Loss) ===
     Exp                                       Name  Hamming Loss
   Exp 5                           OneVsRest LogReg      0.044362
       3 Combined TF-IDF (Word + Char) + Linear SVM      0.044900
       2                   TF-IDF Word + Linear SVM      0.046400
       4         Linear SVM + class_weight=balanced      0.048300
Baseline                           OneVsRest LogReg      0.052181
   Exp 6                OneVsRest LogReg (balanced)      0.052922
   Exp 6                OneVsRest LogReg (balanced)      0.054486
       1          TF-IDF Word + Logistic Regression      0.058900
   Exp 7                      DistilBERT fine-tuned      0.064856
   Exp 7                      DistilBERT fine-tuned      0.066914


In [7]:
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import hamming_loss, f1_score

# Sweep C values around the default of 1.0
C_values = [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]
exp8_results = []

for c in C_values:
    print(f'Training C={c}...')
    clf = OneVsRestClassifier(
        LinearSVC(C=c, max_iter=2000, random_state=SEED), n_jobs=-1
    )
    clf.fit(X_train_combined, y_train)
    y_pred = clf.predict(X_val_combined)
    hl  = hamming_loss(y_val, y_pred)
    f1m = f1_score(y_val, y_pred, average='micro',  zero_division=0)
    f1M = f1_score(y_val, y_pred, average='macro',  zero_division=0)
    exp8_results.append({
        'C': c,
        'Hamming Loss': round(hl,  4),
        'F1-Micro':     round(f1m, 4),
        'F1-Macro':     round(f1M, 4)
    })
    print(f'  → Hamming Loss: {hl:.4f}  F1-Micro: {f1m:.4f}')

exp8_df = pd.DataFrame(exp8_results)
best_c  = exp8_df.loc[exp8_df['Hamming Loss'].idxmin(), 'C']

print(f'\n=== Exp 8 Results ===')
print(exp8_df.to_string(index=False))
print(f'\n Best C value: {best_c}')

Training C=0.01...
  → Hamming Loss: 0.0662  F1-Micro: 0.1813
Training C=0.1...
  → Hamming Loss: 0.0505  F1-Micro: 0.5076
Training C=0.5...
  → Hamming Loss: 0.0456  F1-Micro: 0.5962
Training C=1.0...
  → Hamming Loss: 0.0449  F1-Micro: 0.6171
Training C=5.0...
  → Hamming Loss: 0.0455  F1-Micro: 0.6316
Training C=10.0...
  → Hamming Loss: 0.0459  F1-Micro: 0.6324

=== Exp 8 Results ===
    C  Hamming Loss  F1-Micro  F1-Macro
 0.01        0.0662    0.1813    0.0376
 0.10        0.0505    0.5076    0.3217
 0.50        0.0456    0.5962    0.4634
 1.00        0.0449    0.6171    0.5072
 5.00        0.0455    0.6316    0.5421
10.00        0.0459    0.6324    0.5422

 Best C value: 1.0


In [8]:
from sklearn.calibration import CalibratedClassifierCV


# Model 1 — Exp 3: best SVC (C=1.0)
print('Training Model 1 — LinearSVC C=1.0...')
svc3 = OneVsRestClassifier(
    LinearSVC(C=1.0, max_iter=2000, random_state=SEED), n_jobs=-1
)
svc3.fit(X_train_combined, y_train)
pred_svc3 = svc3.predict(X_val_combined)

# Model 2 — Exp 5: LR with tuned thresholds
print('Training Model 2 — LR + tuned thresholds...')
lr5 = OneVsRestClassifier(
    LogisticRegression(max_iter=2000, solver='liblinear', random_state=SEED)
)
lr5.fit(X_train_combined, y_train)
probs_lr5 = lr5.predict_proba(X_val_combined)

def tune_thresholds(y_true, y_prob,
                    thresholds=None):
    if thresholds is None:
        thresholds = np.arange(0.30, 0.51, 0.05)
    best = []
    for i in range(y_true.shape[1]):
        best_t, best_f1 = 0.5, 0.0
        for t in thresholds:
            pred = (y_prob[:, i] >= t).astype(int)
            f1 = f1_score(y_true[:, i], pred, zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        best.append(best_t)
    return np.array(best)

best_thresholds = tune_thresholds(y_val, probs_lr5)
pred_lr5 = (probs_lr5 >= best_thresholds).astype(int)

# Model 3 — Exp 4: SVM balanced
print('Training Model 3 — SVM balanced...')
svc4 = OneVsRestClassifier(
    LinearSVC(C=1.0, max_iter=2000, class_weight='balanced', random_state=SEED), n_jobs=-1
)
svc4.fit(X_train_combined, y_train)
pred_svc4 = svc4.predict(X_val_combined)

print(' All 3 models trained')

ensemble_pred = ((pred_svc3 + pred_lr5 + pred_svc4) >= 2).astype(int)

hl  = hamming_loss(y_val, ensemble_pred)
f1m = f1_score(y_val, ensemble_pred, average='micro',   zero_division=0)
f1M = f1_score(y_val, ensemble_pred, average='macro',   zero_division=0)
f1s = f1_score(y_val, ensemble_pred, average='samples', zero_division=0)
prec = precision_score(y_val, ensemble_pred, average='micro', zero_division=0)
rec  = recall_score(y_val, ensemble_pred,  average='micro', zero_division=0)

print(f'\n=== Exp 9 — Majority Vote Ensemble ===')
print(f'Hamming Loss : {hl:.4f}')
print(f'F1-Micro     : {f1m:.4f}')
print(f'F1-Macro     : {f1M:.4f}')
print(f'F1-Samples   : {f1s:.4f}')
print(f'Precision    : {prec:.4f}')
print(f'Recall       : {rec:.4f}')
print(f'\nBest so far  : 0.0444 (Exp 5)')
print(f'Ensemble     : {hl:.4f}')
print(f'Improvement  : {0.0444 - hl:.4f}')

Training Model 1 — LinearSVC C=1.0...
Training Model 2 — LR + tuned thresholds...
Training Model 3 — SVM balanced...
 All 3 models trained

=== Exp 9 — Majority Vote Ensemble ===
Hamming Loss : 0.0442
F1-Micro     : 0.6349
F1-Macro     : 0.5222
F1-Samples   : 0.6276
Precision    : 0.7875
Recall       : 0.5319

Best so far  : 0.0444 (Exp 5)
Ensemble     : 0.0442
Improvement  : 0.0002


In [10]:
from sklearn.calibration import CalibratedClassifierCV

print('Calibrating Model 1...')
svc3_cal = OneVsRestClassifier(
    CalibratedClassifierCV(
        LinearSVC(C=1.0, max_iter=2000, random_state=SEED), cv=3
    ), n_jobs=-1
)
svc3_cal.fit(X_train_combined, y_train)
probs_svc3 = svc3_cal.predict_proba(X_val_combined)
print('Calibrating Model 3...')
svc4_cal = OneVsRestClassifier(
    CalibratedClassifierCV(
        LinearSVC(C=1.0, max_iter=2000,
                  class_weight='balanced', random_state=SEED), cv=3
    ), n_jobs=-1
)
svc4_cal.fit(X_train_combined, y_train)
probs_svc4 = svc4_cal.predict_proba(X_val_combined)

# Average probabilities across all 3 models (soft vote)
probs_ensemble = (probs_svc3 + probs_lr5 + probs_svc4) / 3.0

# Tune thresholds on averaged probabilities
print('Tuning per-label thresholds...')
best_t10 = tune_thresholds(y_val, probs_ensemble)
pred_exp10 = (probs_ensemble >= best_t10).astype(int)

hl   = hamming_loss(y_val, pred_exp10)
f1m  = f1_score(y_val, pred_exp10, average='micro',   zero_division=0)
f1M  = f1_score(y_val, pred_exp10, average='macro',   zero_division=0)
f1s  = f1_score(y_val, pred_exp10, average='samples', zero_division=0)
prec = precision_score(y_val, pred_exp10, average='micro', zero_division=0)
rec  = recall_score(y_val, pred_exp10,    average='micro', zero_division=0)

print(f'\n=== Exp 10 — Soft Ensemble + Per-Label Threshold Tuning ===')
print(f'Hamming Loss : {hl:.4f}')
print(f'F1-Micro     : {f1m:.4f}')
print(f'F1-Macro     : {f1M:.4f}')
print(f'F1-Samples   : {f1s:.4f}')
print(f'Precision    : {prec:.4f}')
print(f'Recall       : {rec:.4f}')

print(f'\n=== Final Ranking (all 10 experiments) ===')
final = pd.DataFrame([
    {'Exp': 1,  'Name': 'TF-IDF Word + LR',                    'Hamming Loss': 0.0589},
    {'Exp': 2,  'Name': 'TF-IDF Word + SVM',                   'Hamming Loss': 0.0464},
    {'Exp': 3,  'Name': 'Word+Char TF-IDF + SVM',              'Hamming Loss': 0.0449},
    {'Exp': 4,  'Name': 'SVM balanced',                        'Hamming Loss': 0.0483},
    {'Exp': 5,  'Name': 'LR + tuned thresholds',               'Hamming Loss': 0.0444},
    {'Exp': 6,  'Name': 'Balanced LR + tuned',                 'Hamming Loss': 0.0529},
    {'Exp': 7,  'Name': 'DistilBERT fine-tuned',               'Hamming Loss': 0.0669},
    {'Exp': 8,  'Name': 'SVM C sweep (best C=1.0)',             'Hamming Loss': 0.0449},
    {'Exp': 9,  'Name': 'Majority Vote Ensemble',               'Hamming Loss': 0.0442},
    {'Exp': 10, 'Name': 'Soft Ensemble + Threshold Tuning',     'Hamming Loss': round(hl, 4)},
]).sort_values('Hamming Loss')

print(final.to_string(index=False))

Calibrating Model 1...
Calibrating Model 3...
Tuning per-label thresholds...

=== Exp 10 — Soft Ensemble + Per-Label Threshold Tuning ===
Hamming Loss : 0.0421
F1-Micro     : 0.6820
F1-Macro     : 0.5865
F1-Samples   : 0.6849
Precision    : 0.7517
Recall       : 0.6241

=== Final Ranking (all 10 experiments) ===
 Exp                             Name  Hamming Loss
  10 Soft Ensemble + Threshold Tuning        0.0421
   9           Majority Vote Ensemble        0.0442
   5            LR + tuned thresholds        0.0444
   3           Word+Char TF-IDF + SVM        0.0449
   8         SVM C sweep (best C=1.0)        0.0449
   2                TF-IDF Word + SVM        0.0464
   4                     SVM balanced        0.0483
   6              Balanced LR + tuned        0.0529
   1                 TF-IDF Word + LR        0.0589
   7            DistilBERT fine-tuned        0.0669


In [15]:
print('Generating test predictions...')

probs_svc3_test = svc3_cal.predict_proba(X_test_combined)
probs_lr5_test  = lr5.predict_proba(X_test_combined)
probs_svc4_test = svc4_cal.predict_proba(X_test_combined)

probs_test_ensemble = (probs_svc3_test + probs_lr5_test + probs_svc4_test) / 3.0

# Fixed threshold suited to test probability scale
y_test_pred = (probs_test_ensemble >= 0.15).astype(int)

submission = pd.DataFrame(y_test_pred, columns=ALL_INDICATORS)
submission.insert(0, 'Unique ID', test_ids)

print(f'Submission shape: {submission.shape}')
print(f'Predicted labels per test sample (mean): {y_test_pred.sum(axis=1).mean():.2f}')
print(f'Sample predictions (first 3 rows, first 5 indicators):')
print(submission.iloc[:3, :6])

submission.to_csv('/content/submission.csv', index=False)
submission.to_csv(f'{SAVE_DIR}/../submission.csv', index=False)

print('\n Submission saved')

Generating test predictions...
Submission shape: (998, 28)
Predicted labels per test sample (mean): 0.74
Sample predictions (first 3 rows, first 5 indicators):
   Unique ID  3.1.1 - Maternal mortality ratio  \
0      49848                                 0   
1      52348                                 0   
2     103541                                 0   

   3.1.2 - Proportion of births attended by skilled health personnel  \
0                                                  0                   
1                                                  0                   
2                                                  0                   

   3.2.1 - Under-5 mortality rate  3.2.2 - Neonatal mortality rate  \
0                               0                                0   
1                               0                                0   
2                               0                                0   

   3.3.1 - Number of new HIV infections per 1,000 uninfected populati

In [13]:
print('Ensemble probability stats on test set:')
print(f'  Mean  : {probs_test_ensemble.mean():.4f}')
print(f'  Max   : {probs_test_ensemble.max():.4f}')
print(f'  Median: {np.median(probs_test_ensemble):.4f}')

print('\nTuned thresholds stats:')
print(f'  Mean threshold : {best_t10.mean():.4f}')
print(f'  Min threshold  : {best_t10.min():.4f}')
print(f'  Max threshold  : {best_t10.max():.4f}')

print('\nValidation probability stats:')
print(f'  Mean  : {probs_ensemble.mean():.4f}')
print(f'  Max   : {probs_ensemble.max():.4f}')

Ensemble probability stats on test set:
  Mean  : 0.0358
  Max   : 0.3572
  Median: 0.0238

Tuned thresholds stats:
  Mean threshold : 0.3296
  Min threshold  : 0.3000
  Max threshold  : 0.5000

Validation probability stats:
  Mean  : 0.0718
  Max   : 0.9955


In [19]:
print('Generating test predictions...')

probs_svc3_test = svc3_cal.predict_proba(X_test_combined)
probs_lr5_test  = lr5.predict_proba(X_test_combined)
probs_svc4_test = svc4_cal.predict_proba(X_test_combined)

probs_test_ensemble = (probs_svc3_test + probs_lr5_test + probs_svc4_test) / 3.0

# Threshold 0.10 gives mean labels of 2.24 — closest to training mean of 1.97
y_test_pred = (probs_test_ensemble >= 0.10).astype(int)


submission = pd.DataFrame(y_test_pred, columns=ALL_INDICATORS)
submission.insert(0, 'Unique ID', test_ids)

print(f'Submission shape          : {submission.shape}')
print(f'Mean labels per sample    : {y_test_pred.sum(axis=1).mean():.2f}')
print(f'Min labels per sample     : {y_test_pred.sum(axis=1).min()}')
print(f'Max labels per sample     : {y_test_pred.sum(axis=1).max()}')
print(f'\nPredicted label counts:')
print(submission[ALL_INDICATORS].sum().sort_values(ascending=False).head(10))

# Save locally
submission.to_csv('/content/submission.csv', index=False)

# Save to Drive
submission.to_csv(f'{SAVE_DIR}/../submission.csv', index=False)

print('\n Submission saved — ready to submit')

Generating test predictions...
Submission shape          : (998, 28)
Mean labels per sample    : 2.24
Min labels per sample     : 0
Max labels per sample     : 4

Predicted label counts:
3.b.2 - Total net official development assistance to medical research and basic health sector                                                                                                                                                                                                                                                 984
3.8.1 - Coverage of essential health services (defined as the average coverage of essential services based on tracer interventions that include reproductive, maternal, newborn and child health, infectious diseases, non-communicable diseases and service capacity and access, among the general and the most disadvantaged population)    805
3.4.1 - Mortality rate attributed to cardiovascular disease, cancer, diabetes or chronic respiratory disease                             

In [18]:
import os, shutil

PERSON1_DIR = '/content/drive/MyDrive/SDG_Assignment/results/person1'
os.makedirs(PERSON1_DIR, exist_ok=True)

exp8_df.to_csv(f'{PERSON1_DIR}/experiments_8_10_results.csv', index=False)

shutil.copy('/content/submission.csv', f'{PERSON1_DIR}/submission.csv')

final.to_csv(f'{PERSON1_DIR}/final_ranking_all_experiments.csv', index=False)

import joblib
joblib.dump(svc3_cal, f'{PERSON1_DIR}/best_model_exp10_svc_calibrated.pkl')
joblib.dump(lr5,      f'{PERSON1_DIR}/best_model_exp10_lr.pkl')
joblib.dump(svc4_cal, f'{PERSON1_DIR}/best_model_exp10_svc_balanced.pkl')
print('\nFiles saved:')
for f in sorted(os.listdir(PERSON1_DIR)):
    size = os.path.getsize(f'{PERSON1_DIR}/{f}') / (1024*1024)
    print(f'  {f:<45} {size:.1f} MB')


Files saved:
  best_model_exp10_lr.pkl                       16.5 MB
  best_model_exp10_svc_balanced.pkl             49.5 MB
  best_model_exp10_svc_calibrated.pkl           49.5 MB
  experiments_8_10_results.csv                  0.0 MB
  final_ranking_all_experiments.csv             0.0 MB
  submission.csv                                0.1 MB
